<a href="https://colab.research.google.com/github/vitorpantoja12/Projeto-uirapuru/blob/main/Projeto_Uirapuru_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PROJETO UIRAPURU AI**



In [4]:
!git clone https://github.com/vitorpantoja12/Projeto-uirapuru.git
!ls /content/Projeto-uirapuru/dataset/Treino/

Cloning into 'Projeto-uirapuru'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 59 (delta 4), reused 49 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 6.50 MiB | 11.64 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Uirapuru-da-guiana


In [6]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")

Usando: cuda


In [13]:
import requests

url = "https://api.inaturalist.org/v1/observations"

params = {
    "taxon_name": "Rostrhamus sociabilis",
    "photos": "true",
    "quality_grade": "research",
    "swlat": -56,
    "swlng": -82,
    "nelat": 13,
    "nelng": -34,
    "per_page": 10,
    "order_by": "created_at",
    "order": "desc"
}

response = requests.get(url, params=params)

print(response.status_code)
dados = response.json()
print(dados.keys())
print(dados["total_results"])
print(dados["results"][0])

200
dict_keys(['total_results', 'page', 'per_page', 'results'])
8256
{'quality_grade': 'research', 'time_observed_at': '2026-01-25T10:10:00-04:00', 'taxon_geoprivacy': 'open', 'annotations': [{'uuid': '330289a7-b9d4-42d6-94c5-0c532d56f685', 'controlled_attribute_id': 17, 'controlled_value_id': 18, 'concatenated_attr_val': '17|18', 'user_id': 9214932, 'votes': [], 'user': {'id': 9214932, 'login': 'olmagon', 'spam': False, 'suspended': False, 'created_at': '2025-04-27T00:00:39+00:00', 'login_autocomplete': 'olmagon', 'login_exact': 'olmagon', 'name': None, 'name_autocomplete': None, 'orcid': None, 'icon': 'https://static.inaturalist.org/attachments/users/icons/9214932/c3ad683334a7bcb34192769314cfd967-thumb.jpg?1746010069', 'observations_count': 10821, 'identifications_count': 80483, 'journal_posts_count': 0, 'activity_count': 91304, 'species_count': 2448, 'annotated_observations_count': 66945, 'universal_search_rank': 10821, 'roles': [], 'site_id': 1, 'icon_url': 'https://static.inatural

In [16]:
import os

BASE_DIR = "/content/Projeto-uirapuru/dataset/treino"

species = "Rostrhamus_sociabilis"

save_dir = os.path.join(BASE_DIR, species)

os.makedirs(save_dir, exist_ok=True)

print(save_dir)

/content/Projeto-uirapuru/dataset/treino/Rostrhamus_sociabilis


In [9]:
# Caminhos do dataset
TRAIN_DIR = '/content/Projeto-uirapuru/dataset/Treino'
VAL_DIR   = '/content/Projeto-uirapuru/dataset/Validação'

# Transformações para treinamento
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Transformações para validação
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Carrega os conjuntos
train_set = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=transform_train
)

val_set = datasets.ImageFolder(
    root=VAL_DIR,
    transform=transform_val
)

# DataLoaders
train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

# Classes
classes = train_set.classes

print(f"Classes encontradas ({len(classes)}):")
for i, classe in enumerate(classes):
    print(f"{i:2d} - {classe}")

print(f"\nTreino: {len(train_set)} imagens")
print(f"Validação: {len(val_set)} imagens")

FileNotFoundError: Found no valid file for the classes Uirapuru-da-guiana. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [ ]:
NUM_CLASSES = len(classes)

modelo = models.resnet18(weights='IMAGENET1K_V1')

# Congela todas as camadas base
for param in modelo.parameters():
    param.requires_grad = False

# Substitui a camada final pelo número de espécies do projeto
modelo.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(modelo.fc.in_features, NUM_CLASSES)
)

modelo = modelo.to(device)
print(f"Modelo pronto — {NUM_CLASSES} classes")

In [ ]:
criterio = nn.CrossEntropyLoss()
otimizador = torch.optim.Adam(modelo.fc.parameters(), lr=0.001)

historico = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

EPOCHS = 15

for epoch in range(EPOCHS):
    # Treino
    modelo.train()
    loss_train, acertos, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        otimizador.zero_grad()
        saidas = modelo(imgs)
        loss = criterio(saidas, labels)
        loss.backward()
        otimizador.step()
        loss_train += loss.item()
        acertos += (saidas.argmax(1) == labels).sum().item()
        total += labels.size(0)
    acc_train = acertos / total

    # Validação
    modelo.eval()
    loss_val, acertos_val, total_val = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            saidas = modelo(imgs)
            loss_val += criterio(saidas, labels).item()
            acertos_val += (saidas.argmax(1) == labels).sum().item()
            total_val += labels.size(0)
    acc_val = acertos_val / total_val

    historico['train_acc'].append(acc_train)
    historico['val_acc'].append(acc_val)
    historico['train_loss'].append(loss_train / len(train_loader))
    historico['val_loss'].append(loss_val / len(val_loader))

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Loss treino: {loss_train/len(train_loader):.3f} | "
          f"Acc treino: {acc_train:.2%} | "
          f"Acc val: {acc_val:.2%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(historico['train_acc'], label='Treino')
ax1.plot(historico['val_acc'], label='Validação')
ax1.set_title('Acurácia')
ax1.set_xlabel('Época')
ax1.legend()

ax2.plot(historico['train_loss'], label='Treino')
ax2.plot(historico['val_loss'], label='Validação')
ax2.set_title('Loss')
ax2.set_xlabel('Época')
ax2.legend()

plt.tight_layout()
plt.show()